# Credit Risk Scorecard Development
## PD_cust × f(deal) Interaction Model — Development Notebook

**Project:** 
**Analyst:** 
**Date:** 
**Model version:** 

---

### Notebook Purpose

This notebook follows the six-phase scorecard development process for a car finance PD model. Each phase contains code cells (ready to run), markdown cells (to be completed with analyst findings), and decision checkpoint cells that document which path was taken and why.

**How to use this notebook:**
- Run code cells in sequence within each phase
- Fill in all `[ANALYST NOTE]` sections with your findings before moving to the next phase
- Complete all decision checkpoint cells — these become the governance audit trail
- Do not skip phases; each phase informs the configuration of the next

---

| Phase | Purpose | Key Output |
|-------|---------|------------|
| 0 | Data Setup & Quality | Clean dev and OOT samples |
| 1 | Variable Assessment & Binning | IV table, transformation decisions |
| 2 | Multiplicative Assumption Testing | BD results, interaction evidence |
| 3 | Model Configuration & Building | Fitted interaction models |
| 4 | Model Comparison & Selection | Selected model with rationale |
| 5 | Validation | Gini, KS, HL, PSI, CSI |
| 6 | Scorecard Scaling & Output | Points table, governance pack |


## Setup


In [ ]:
# ── Dependencies ────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# Scorecard pipeline components
from data.extractor import DataExtractor
from preprocessing.binning import BinningPipeline
from testing.statistical_tests import (
    InteractionTestingPipeline, ContingencyTable,
    PairwiseBreslowDay, DealVariableDiagnostics,
)
from modelling.logistic_model import ScorecardLogisticRegression
from modelling.scorecard_scaler import ScorecardScaler
from modelling.interaction_model import (
    InteractionLogisticRegression, DealVariableConfig,
)
from modelling.model_comparison import ModelComparison
from modelling.deal_variable_analysis import DealVariableLogOddsAnalysis
from modelling.deal_variable_plots import DealVariablePlotter
from pipeline import ScorecardPipeline, DataSplitConfig
from interaction_pipeline import InteractionScorecardPipeline
from validation.metrics import ValidationReport

print('All imports successful')


### Configuration

Set all project-level parameters here. This cell is the single source of truth for variable lists, date ranges, and connection details.


In [ ]:
# ── Connection ──────────────────────────────────────────────────────────────
CONNECTION_STRING = "mssql+pyodbc://server/database?driver=ODBC+Driver+17+for+SQL+Server"

# ── Date ranges ──────────────────────────────────────────────────────────────
DEV_START = "2020-01-01"
DEV_END   = "2022-12-31"
OOT_START = "2023-01-01"
OOT_END   = "2023-12-31"

# ── Target variable ──────────────────────────────────────────────────────────
TARGET = "default_flag"   # 1 = bad, 0 = good

# ── Variables ────────────────────────────────────────────────────────────────
CUSTOMER_VARS = [
    "annual_income",
    "months_at_address",
    "employment_status",
    # add / remove as needed
]

DEAL_VARS = [
    "ltv_ratio",
    "deposit_pct",
    "loan_term_months",
    "instalment_amount",
    # add / remove as needed
]

EQUIFAX_COL = "equifax_score"   # Proxy for PD_cust until full customer scorecard built

# ── Scorecard scaling ────────────────────────────────────────────────────────
PDO        = 20
BASE_SCORE = 600
BASE_ODDS  = 50

# ── Output directories ───────────────────────────────────────────────────────
PLOT_DIR = "outputs/plots"
DATA_DIR = "outputs/data"

import os
os.makedirs(PLOT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

print('Configuration set')


---
## Phase 0 — Data Setup and Quality

**Purpose:** Establish clean, representative development and OOT samples before any modelling. Data quality issues identified here must be resolved before proceeding — problems in this phase compound through every subsequent step.

**Key checks:**
- Record counts and bad rates (dev vs OOT should be broadly comparable)
- Missing rates per variable
- Date distribution confirms no data leakage
- No post-default variables inadvertently included


### 0.1 Data Extraction


In [ ]:
pipeline = ScorecardPipeline(
    connection_string  = CONNECTION_STRING,
    target             = TARGET,
    customer_variables = CUSTOMER_VARS,
    deal_variables     = DEAL_VARS,
    pdo                = PDO,
    base_score         = BASE_SCORE,
    base_odds          = BASE_ODDS,
)

split_config = DataSplitConfig.from_dates(
    dev_start = DEV_START, dev_end = DEV_END,
    oot_start = OOT_START, oot_end = OOT_END,
)

pipeline.extract_data(split_config)

dev_df = pipeline.dev_data
oot_df = pipeline.oot_data

print(f'Development: {len(dev_df):,} records | Bad rate: {dev_df[TARGET].mean():.2%}')
print(f'OOT:         {len(oot_df):,} records | Bad rate: {oot_df[TARGET].mean():.2%}')


### 0.2 Data Quality Checks


In [ ]:
# ── Missing rates ────────────────────────────────────────────────────────────
all_vars = CUSTOMER_VARS + DEAL_VARS + [EQUIFAX_COL, TARGET]
missing_dev = dev_df[all_vars].isnull().mean().rename('missing_pct_dev')
missing_oot = oot_df[all_vars].isnull().mean().rename('missing_pct_oot')
missing_summary = pd.concat([missing_dev, missing_oot], axis=1)
missing_summary['flag'] = missing_summary['missing_pct_dev'].apply(
    lambda x: '⚠️  >30%' if x > 0.30 else ('⚡ >10%' if x > 0.10 else '✅')
)
print(missing_summary.to_string())


In [ ]:
# ── Bad rate by origination month ────────────────────────────────────────────
# Adjust 'origination_date' to the actual date column name in your data

DATE_COL = 'origination_date'  # <-- update this

fig, ax = plt.subplots(figsize=(13, 4))

for df, label, color in [
    (dev_df, 'Development', '#2980b9'),
    (oot_df, 'OOT',         '#e67e22'),
]:
    monthly = (
        df.assign(month=pd.to_datetime(df[DATE_COL]).dt.to_period('M'))
        .groupby('month')[TARGET].mean()
    )
    ax.plot(monthly.index.astype(str), monthly.values,
            label=label, color=color, marker='o', markersize=3)

ax.axvline(x=DEV_END, color='#c0392b', linestyle='--', linewidth=1, label='Dev/OOT split')
ax.set_title('Bad Rate by Origination Month')
ax.set_ylabel('Bad Rate')
ax.legend()
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/phase0_bad_rate_monthly.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Descriptive statistics ───────────────────────────────────────────────────
print('Development sample:')
print(dev_df[DEAL_VARS + [EQUIFAX_COL]].describe().round(2))


### Phase 0 — Conclusions and Sign-off

> **[ANALYST NOTE]** Complete this section before proceeding to Phase 1.

**Record counts:**
- Development: *[N records, N bads, bad rate X%]*
- OOT: *[N records, N bads, bad rate X%]*
- Bad rate difference: *[X pp — acceptable / requires documentation]*

**Missing data:**
- Variables with > 10% missing: *[list or 'None']*
- Action taken: *[imputation strategy / treated as separate bin / excluded]*

**Data leakage check:**
- *[Confirm all variables are available at point of origination]*

**Decision:** ☐ Proceed to Phase 1 &nbsp;&nbsp; ☐ Resolve data issues first

**Issues to resolve before Phase 1 (if any):**
- *[List any outstanding issues]*


---
## Phase 1 — Variable Assessment and Binning

**Purpose:** Understand each variable's relationship with default, select candidates for modelling, and determine the appropriate input representation (WoE-binned, raw continuous, or transformed continuous). Phase 1 directly informs `DealVariableConfig` choices in Phase 3.


### 1.1 WoE Binning and IV Calculation

Start with the cut points below. Refine iteratively until:
- No bin has < 5% of the population
- No bin has < 50 bads
- WoE is monotonic for continuous variables (or a documented exception applies)


In [ ]:
# ── Cut point specification ───────────────────────────────────────────────────
# Update cut points based on quantile analysis and business knowledge
# Leave a variable out of this dict to use equal-frequency auto-binning

CUSTOMER_CUT_POINTS = {
    'annual_income':     [15000, 25000, 40000, 65000],
    'months_at_address': [6, 24, 60],
    # 'employment_status' → categorical, handled via var_types
}

DEAL_CUT_POINTS = {
    'ltv_ratio':         [60, 80, 100, 110],
    'deposit_pct':       [5, 10, 15, 20],
    'loan_term_months':  [24, 36, 48],
    'instalment_amount': [200, 350, 500],
}

CUSTOMER_VAR_TYPES = {
    'employment_status': 'categorical',
}

pipeline.run_binning(
    customer_cut_points = CUSTOMER_CUT_POINTS,
    deal_cut_points     = DEAL_CUT_POINTS,
    customer_var_types  = CUSTOMER_VAR_TYPES,
)

dev_woe = pipeline.dev_woe
oot_woe = pipeline.oot_woe


In [ ]:
# ── IV bar chart ─────────────────────────────────────────────────────────────
iv_all = pd.concat([
    pipeline.customer_binner.iv_summary.assign(component='Customer'),
    pipeline.deal_binner.iv_summary.assign(component='Deal'),
])

color_map = {
    'Useless': '#bdc3c7', 'Weak': '#e67e22', 'Medium': '#2980b9',
    'Strong': '#27ae60', 'Suspiciously Strong — check for data leakage': '#c0392b',
}

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(
    iv_all['variable'], iv_all['iv'],
    color=[color_map.get(r, '#7f8c8d') for r in iv_all['rating']],
    edgecolor='white', linewidth=0.5,
)
ax.axvline(0.10, color='#e67e22', linestyle='--', linewidth=1, label='0.10 — Weak')
ax.axvline(0.30, color='#2980b9', linestyle='--', linewidth=1, label='0.30 — Medium')
ax.axvline(0.50, color='#c0392b', linestyle='--', linewidth=1, label='0.50 — Suspicious')
ax.set_xlabel('Information Value (IV)')
ax.set_title('IV Summary — All Candidate Variables')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/phase1_iv_summary.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── WoE profile plots — one per variable ─────────────────────────────────────
vars_to_plot = DEAL_VARS   # change to CUSTOMER_VARS or a specific subset as needed

for var in vars_to_plot:
    binner = pipeline.deal_binner
    if var not in binner._configs:
        continue
    bs = binner.get_bin_stats(var)
    if bs is None or bs.empty:
        continue

    fig, ax1 = plt.subplots(figsize=(10, 4))
    ax2 = ax1.twinx()

    x = np.arange(len(bs))
    ax2.bar(x, bs['bad_rate'], color='#bdc3c7', alpha=0.5, label='Bad Rate')
    ax1.plot(x, bs['woe'], color='#2980b9', linewidth=2,
             marker='o', markersize=5, label='WoE', zorder=3)
    ax1.axhline(0, color='#95a5a6', linewidth=0.8, linestyle=':')

    ax1.set_xticks(x)
    ax1.set_xticklabels([str(b) for b in bs['bin']], rotation=35, ha='right', fontsize=7)
    ax1.set_ylabel('WoE', color='#2980b9')
    ax2.set_ylabel('Bad Rate', color='#7f8c8d')
    ax1.set_title(f'WoE Profile: {var}  |  IV = {binner._configs[var]["binner"].iv:.3f}')
    ax1.set_zorder(ax2.get_zorder() + 1)
    ax1.patch.set_visible(False)

    issues = binner._configs[var]['binner'].validation_issues
    if issues:
        ax1.set_xlabel(f'⚠️  {issues[0]}', fontsize=7, color='#e74c3c')

    plt.tight_layout()
    plt.savefig(f'{PLOT_DIR}/phase1_woe_{var}.png', dpi=150, bbox_inches='tight')
    plt.show()


### 1.1 — Binning Conclusions

> **[ANALYST NOTE]** Record IV results and cut point decisions for each variable.

**Variables selected (IV 0.10–0.50):**
- Customer: *[list]*
- Deal: *[list]*

**Variables excluded:**
- *[Variable]* — IV = *[X]* — Reason: *[Useless / Leakage suspected / Business decision]*

**Cut point revisions made:**
- *[Variable]* — *[Original cuts]* → *[Revised cuts]* — Reason: *[bin too small / non-monotonic / etc.]*

**WoE monotonicity exceptions:**
- *[Variable]* — Non-monotonic accepted because: *[business rationale]*


### 1.2 Log-Odds Analysis and Transformation Decisions

This step determines whether deal variables need transformation before entering the interaction model. Results directly determine the `mode` and `transform` fields in `DealVariableConfig` (Phase 3). Run for all deal variables including those that may use WoE — the by-band plots also provide early evidence for or against interaction terms.


In [ ]:
interaction_pipeline = InteractionScorecardPipeline(
    target         = TARGET,
    equifax_col    = EQUIFAX_COL,
    deal_variables = DEAL_VARS,
    pdo            = PDO,
    base_score     = BASE_SCORE,
    base_odds      = BASE_ODDS,
)

# Run log-odds analysis — generates 4-panel plot per variable + model comparison summary
logodds_output = interaction_pipeline.log_odds_analysis(
    df                   = dev_df,
    n_bands              = 4,
    band_labels          = ['Sub-Prime', 'Near-Prime', 'Prime', 'Super-Prime'],
    run_model_comparison = True,
    plot                 = True,
    plot_save_dir        = f'{PLOT_DIR}/phase1_logodds',
)

logodds_results    = logodds_output['log_odds_results']
logodds_model_comp = logodds_output['model_comparison']


In [ ]:
# ── Transformation and interaction evidence summary ───────────────────────────
summary_rows = []
for var, res in logodds_results.items():
    summary_rows.append({
        'variable':              var,
        'linearity':             res.linearity_flag,
        'spearman_overall':      round(res.spearman_overall, 3),
        'transform_type':        res.transform_type,
        'interaction_evidence':  res.interaction_evidence[:80] + '...',
    })

print(pd.DataFrame(summary_rows).to_string(index=False))
print()
print('With vs Without Interaction — model comparison:')
print(logodds_model_comp.to_string(index=False))


### 1.2 — Transformation and Input Mode Decisions

Use the log-odds shape plots and the table above to complete this decision record. These decisions carry forward directly into `DealVariableConfig` in Phase 3.

> **[ANALYST NOTE]** Complete the table below for every deal variable.

| Variable | Log-odds shape | Transform needed | Input mode | Interaction evidence | Decision |
|----------|---------------|-----------------|------------|---------------------|----------|
| ltv_ratio | *[Linear / Monotonic / Non-linear]* | *[none / log / sqrt / split]* | *[woe / continuous]* | *[Strong / Moderate / Weak]* | *[WoE / Continuous / Split]* |
| deposit_pct | | | | | |
| loan_term_months | | | | | |
| instalment_amount | | | | | |

**Early interaction evidence summary:**
- Variables with diverging log-odds profiles across Equifax bands: *[list]*
- Variables where LR test is significant (p < 0.05): *[list]*
- Variables where log-odds profiles are approximately parallel: *[list]*


---
## Phase 2 — Multiplicative Assumption Testing (Breslow-Day)

**Purpose:** Formally test whether `PD = PD_cust × f(deal)` is valid. A BD failure is a finding, not an error — it confirms the need for the interaction model built in Phase 3. Document the result and root cause for every variable.

**Breslow-Day logic:**
- H₀: The odds ratio between each deal variable and default is consistent across Equifax strata
- p > 0.07 → PASS → Run CMH to confirm independent predictive power
- p 0.05–0.07 → BORDERLINE → Run pairwise BD; assess practical significance
- p < 0.05 → FAIL → Run stratum diagnostics to identify root cause


In [ ]:
pipeline.run_interaction_testing(
    strata_variable = EQUIFAX_COL,
    n_strata        = 3,   # Low / Medium / High
)


In [ ]:
# ── Odds ratio forest plot ────────────────────────────────────────────────────
# Visualises per-stratum OR with 95% CI for each deal variable

bd_results = pipeline.interaction_tester._results
strata     = ['Low', 'Medium', 'High']
colors    = ['#c0392b', '#e67e22', '#27ae60']

fig, axes = plt.subplots(
    1, len(DEAL_VARS),
    figsize=(4 * len(DEAL_VARS), 5),
    sharey=True,
)

if len(DEAL_VARS) == 1:
    axes = [axes]

for ax, var in zip(axes, DEAL_VARS):
    if var not in bd_results:
        ax.set_title(f'{var}\n(not tested)')
        continue

    res = bd_results[var].breslow_day
    for i, stratum in enumerate(strata):
        or_val = res.odds_ratios.get(stratum, np.nan)
        ci     = res.confidence_intervals.get(stratum, (np.nan, np.nan))
        ax.errorbar(
            or_val, i,
            xerr=[[or_val - ci[0]], [ci[1] - or_val]],
            fmt='o', color=colors[i], markersize=7, capsize=4,
        )

    ax.axvline(res.common_odds_ratio, color='#7f8c8d',
               linestyle='--', linewidth=1, label=f'MH OR={res.common_odds_ratio:.2f}')
    ax.axvline(1.0, color='#2c3e50', linewidth=0.8, linestyle=':')
    ax.set_yticks(range(len(strata)))
    ax.set_yticklabels(strata)
    ax.set_xlabel('Odds Ratio (95% CI)')
    p_eff = res.effective_p_value
    flag  = '✅' if p_eff > 0.07 else ('⚡' if p_eff > 0.05 else '❌')
    ax.set_title(f'{var}\nBD p={p_eff:.3f} {flag}\n{res.decision.value}', fontsize=8)
    ax.legend(fontsize=7)

fig.suptitle('Breslow-Day: Odds Ratios by Equifax Stratum', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/phase2_bd_forest_plot.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Breslow-Day summary table ─────────────────────────────────────────────────
print(pipeline.interaction_tester.summary().to_string(index=False))


In [ ]:
# ── Stratum diagnostics for BD failures ──────────────────────────────────────
# Run this cell only if one or more variables have BD FAIL
# It identifies whether the failure is driven by sparsity or genuine interaction

pipeline.run_diagnostics(
    strata_variable    = EQUIFAX_COL,
    n_strata           = 3,
    min_cell_threshold = 5,
    verbose            = True,
)


### Phase 2 — Conclusions and Decision

> **[ANALYST NOTE]** Record BD results for every deal variable and state the decision.

**Breslow-Day Results:**

| Variable | BD p-value | Tarone applied | Decision | Root cause (if FAIL) |
|----------|-----------|----------------|----------|---------------------|
| ltv_ratio | *[p]* | *[Yes/No]* | *[PASS/BORDERLINE/FAIL]* | *[Genuine interaction / Sparsity / Artefact]* |
| deposit_pct | | | | |
| loan_term_months | | | | |
| instalment_amount | | | | |

**CMH Results (for variables that passed BD):**
- *[Variable]* — CMH p = *[X]* — *[Include / Exclude]*

**Overall structure decision:**

☐ All variables pass BD → Standard scorecard sufficient (skip Phase 3 interaction model)

☐ One or more variables fail BD → Proceed to interaction model (Phase 3)

**Interaction model justified for:** *[list variables]*

**Practical significance assessment:**
- OR range across strata: *[small <0.2 / moderate 0.2–0.5 / large >0.5 on log scale]*
- Business interpretation: *[explain why interaction makes sense commercially]*


---
## Phase 3 — Model Configuration and Building

**Purpose:** Configure each deal variable based on Phase 1 and Phase 2 findings, then fit models across all term structures for governance comparison. This phase translates analytical findings into concrete model configurations.


### 3.1 Deal Variable Configuration

Complete this based on the decisions recorded in Phase 1 (transformation) and Phase 2 (interaction evidence). Each variable's `mode`, `transform`, `include_main`, and `include_interaction` should be directly traceable to a Phase 1 or Phase 2 finding.


In [ ]:
# ── Per-variable configuration ────────────────────────────────────────────────
# Update each variable based on Phase 1 log-odds analysis and Phase 2 BD results
# mode:      'woe'        → reads {var}_woe column
#            'continuous' → reads raw {var} column, applies transform
# transform: 'none' | 'log' | 'sqrt'  (continuous mode only)
# include_main:        True/False → whether β_j × input_j is in the model
# include_interaction: True/False → whether β_ij × (Equifax × input_j) is in the model

DEAL_CONFIGS = {
    'ltv_ratio': DealVariableConfig(
        mode                = 'continuous',   # log-odds was monotonic → continuous
        transform           = 'log',          # curved relationship → log transform
        include_main        = True,
        include_interaction = True,           # BD FAIL with genuine interaction
    ),
    'deposit_pct': DealVariableConfig(
        mode                = 'continuous',
        transform           = 'none',         # approximately linear in log-odds space
        include_main        = True,
        include_interaction = True,
    ),
    'loan_term_months': DealVariableConfig(
        mode                = 'woe',          # non-monotonic → WoE handles shape
        include_main        = True,
        include_interaction = True,
    ),
    'instalment_amount': DealVariableConfig(
        mode                = 'continuous',
        transform           = 'log',
        include_main        = True,
        include_interaction = True,
    ),
}

# Preview the configuration before fitting
from modelling.interaction_model import InteractionLogisticRegression
preview_model = InteractionLogisticRegression(
    equifax_col    = EQUIFAX_COL,
    deal_variables = DEAL_VARS,
    target         = TARGET,
    deal_configs   = DEAL_CONFIGS,
)
print(preview_model.mode_summary().to_string(index=False))


### 3.2 Binning (WoE-mode variables only)


In [ ]:
# Run binning only for WoE-mode variables
# Continuous-mode variables do not need WoE transformation

woe_vars = [v for v, cfg in DEAL_CONFIGS.items() if cfg.mode == 'woe']
print(f'WoE-mode variables requiring binning: {woe_vars}')

woe_cut_points = {v: DEAL_CUT_POINTS[v] for v in woe_vars if v in DEAL_CUT_POINTS}

interaction_pipeline.run_binning(
    df_dev          = dev_df,
    df_oot          = oot_df,
    deal_cut_points = woe_cut_points,
    deal_configs    = DEAL_CONFIGS,
)


### 3.3 Fit All Term Structures

Fits three parallel sets of models for governance comparison:
- **Full:** Equifax main + deal main + interaction *(statistically correct)*
- **Interaction only:** Equifax main + interaction, no deal main *(⚠️ hierarchical violation — for comparison only)*
- **Main only:** Equifax main + deal main, no interaction *(baseline)*


In [ ]:
interaction_pipeline.fit_all_term_structures(
    max_combo_size = 2,    # singles and pairs; increase for larger variable sets
    min_iv         = 0.10,
    max_iv         = 0.50,
    deal_configs   = DEAL_CONFIGS,
)

print(f'Total models fitted: {len(interaction_pipeline.models)}')
print('Models:', interaction_pipeline.model_names)


### Phase 3 — Configuration Sign-off

> **[ANALYST NOTE]** Confirm all model configurations are correct and trace each decision.

**Configuration rationale:**
- *[Variable]* set to continuous/log because: *[Phase 1 log-odds showed curved monotonic relationship]*
- *[Variable]* set to WoE because: *[Phase 1 log-odds non-monotonic — WoE handles shape via bins]*
- *[Variable]* interaction included because: *[Phase 2 BD p = X, genuine interaction confirmed]*

**Models fitted:**
- Full models: *[N]*
- Interaction-only models: *[N]* *(for comparison only)*
- Main-only models: *[N]* *(baseline)*

**Any models removed before comparison:** *[list and reason]*


---
## Phase 4 — Model Comparison and Selection

**Purpose:** Select the best model using statistical criteria, discrimination metrics, and business judgement. Produce governance-ready comparison outputs. Model selection rationale must be documented and traceable.


In [ ]:
# ── Full model comparison ─────────────────────────────────────────────────────
interaction_pipeline.compare_models(criterion='aic')


In [ ]:
# ── Point contribution breakdown (at P25 / P50 / P75 Equifax) ────────────────
contrib_df = interaction_pipeline.contribution_report(print_table=True)
contrib_df.to_csv(f'{DATA_DIR}/phase4_contributions.csv', index=False)


In [ ]:
# ── Marginal deal effect at Equifax percentiles ───────────────────────────────
marginal_df = interaction_pipeline.marginal_effect_report()
marginal_df.to_csv(f'{DATA_DIR}/phase4_marginal_effects.csv', index=False)


In [ ]:
# ── Score distribution plot for the selected model ───────────────────────────
# Update 'selected_model' after reviewing the comparison above

selected_model = interaction_pipeline.best_model  # override after review if needed
print(f'Best model by AIC: {selected_model.model_name}')
print(f'Term structure:     {selected_model.term_structure}')

y_pred_dev = selected_model.predict_proba(interaction_pipeline.dev_woe)
y_pred_oot = selected_model.predict_proba(interaction_pipeline.oot_woe)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, y_true, y_pred, label in [
    (axes[0], dev_df[TARGET], y_pred_dev, 'Development'),
    (axes[1], oot_df[TARGET], y_pred_oot, 'OOT'),
]:
    bads  = y_pred[y_true == 1]
    goods = y_pred[y_true == 0]
    ax.hist(goods, bins=40, alpha=0.6, color='#27ae60', label='Goods', density=True)
    ax.hist(bads,  bins=40, alpha=0.6, color='#c0392b', label='Bads',  density=True)
    ax.set_title(f'PD Distribution — {label}')
    ax.set_xlabel('Predicted PD')
    ax.set_ylabel('Density')
    ax.legend()

plt.suptitle(f'Score Separation — {selected_model.model_name}', fontsize=10)
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/phase4_score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── ROC curves: development vs OOT ───────────────────────────────────────────
from sklearn.metrics import roc_curve, roc_auc_score

fig, ax = plt.subplots(figsize=(6, 6))

for y_true, y_pred, label, color in [
    (dev_df[TARGET], y_pred_dev, 'Development', '#2980b9'),
    (oot_df[TARGET], y_pred_oot, 'OOT',         '#e67e22'),
]:
    fpr, tpr, _ = roc_curve(y_true, y_pred)
    auc = roc_auc_score(y_true, y_pred)
    gini = 2 * auc - 1
    ax.plot(fpr, tpr, color=color, linewidth=2,
            label=f'{label}  Gini={gini:.3f}')

ax.plot([0, 1], [0, 1], color='#95a5a6', linestyle='--', linewidth=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title(f'ROC Curve — {selected_model.model_name}')
ax.legend()
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/phase4_roc.png', dpi=150, bbox_inches='tight')
plt.show()


### Phase 4 — Model Selection Decision

**Selection criteria checklist:**

| Criterion | Threshold | Result | Pass? |
|-----------|-----------|--------|-------|
| LR test (full vs main-only) | p < 0.05 | *[p = X]* | ☐ |
| Gini — Development | Target 0.45–0.65 | *[X]* | ☐ |
| Gini drop (dev → OOT) | ≤ 0.05 | *[X]* | ☐ |
| ΔAIC vs best model | < 2 for equivalence | *[X]* | ☐ |
| All interaction p-values | < 0.05 | *[list any exceptions]* | ☐ |
| Coefficient signs correct | All positive direction | *[any reversals?]* | ☐ |
| Business interpretable | Qualitative | *[assessment]* | ☐ |

> **[ANALYST NOTE]** State the selected model and rationale.

**Selected model:** *[model name]*

**Rationale:** *[why this model was selected over alternatives — reference the criteria table]*

**Models considered but not selected:**
- *[model name]* — excluded because: *[reason]*

**Governance note on interaction-only models:**
- Interaction-only models were assessed for comparison as requested
- These violate the hierarchical principle and have not been selected for deployment
- Comparison result: *[full model outperformed / equivalent / note any unusual findings]*


---
## Phase 5 — Model Validation

**Purpose:** Confirm the selected model is stable, well-calibrated, and performs comparably on the unseen OOT sample. All validation metrics must meet thresholds before governance sign-off.


In [ ]:
# Full validation suite
# Validate the selected interaction model, not the standalone customer scorecard.

interaction_pipeline.validate(model=selected_model)


In [ ]:
# ── Calibration: observed vs predicted by score band ─────────────────────────
from validation.metrics import CalibrationMetrics

calib = CalibrationMetrics(dev_df[TARGET], y_pred_dev)
hl    = calib.hosmer_lemeshow()
obs_pred = calib.observed_vs_predicted(n_bands=10)

print(f"Hosmer-Lemeshow: χ²={hl['statistic']:.4f}, p={hl['p_value']:.4f}")
print(hl['recommendation'])
print()
print(obs_pred[['band', 'n_total', 'observed_bad_rate', 'predicted_pd', 'pct_difference']].to_string(index=False))


In [ ]:
# ── Calibration plot ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))

obs_pred_plot = obs_pred.copy()
x = range(len(obs_pred_plot))
ax.plot(x, obs_pred_plot['observed_bad_rate'], color='#c0392b',
        marker='o', linewidth=2, label='Observed bad rate')
ax.plot(x, obs_pred_plot['predicted_pd'], color='#2980b9',
        marker='s', linewidth=2, linestyle='--', label='Predicted PD')
ax.set_xticks(x)
ax.set_xticklabels([str(b) for b in obs_pred_plot['band']], rotation=35, ha='right', fontsize=7)
ax.set_ylabel('Rate')
ax.set_title(
    f'Observed vs Predicted Bad Rate by Score Band\n'
    f"Hosmer-Lemeshow p={hl['p_value']:.4f} — "
    f"{'Well calibrated' if hl['well_calibrated'] else 'Recalibration required'}"
)
ax.legend()
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/phase5_calibration.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Stability: PSI and CSI ───────────────────────────────────────────────────
from validation.metrics import StabilityMetrics

psi_result = StabilityMetrics.calculate_psi(
    expected = pd.Series(y_pred_dev),
    actual   = pd.Series(y_pred_oot),
    label    = 'PD Score PSI',
)
print(f"PSI: {psi_result['psi']:.4f} — {psi_result['status']}")
print(f"Action: {psi_result['action']}")

# CSI across deal variables
csi_vars = [f'{v}_woe' for v in DEAL_VARS if f'{v}_woe' in dev_df.columns] + \
           [v for v, cfg in DEAL_CONFIGS.items() if cfg.mode == 'continuous']

csi_df = StabilityMetrics.run_csi_all(
    expected_df = interaction_pipeline.dev_woe,
    actual_df   = interaction_pipeline.oot_woe,
    variables   = csi_vars,
)
print()
print(csi_df.to_string(index=False))


### Phase 5 — Validation Conclusions

| Metric | Value | Threshold | Status |
|--------|-------|-----------|--------|
| Gini — Development | *[X]* | 0.45–0.65 | ☐ Pass ☐ Flag |
| Gini — OOT | *[X]* | Dev − 0.05 | ☐ Pass ☐ Flag |
| KS Statistic | *[X]* | > 0.30 typical | ☐ Pass ☐ Flag |
| Hosmer-Lemeshow p | *[X]* | > 0.05 | ☐ Pass ☐ Flag |
| PSI (score) | *[X]* | < 0.10 stable | ☐ Pass ☐ Flag |
| CSI (max variable) | *[X]* | < 0.10 stable | ☐ Pass ☐ Flag |

> **[ANALYST NOTE]** Complete the table and record any remediation actions.

**Issues identified:**
- *[Metric]* failed threshold because: *[reason]*
- Action taken: *[recalibration / model revision / documented exception]*

**Decision:** ☐ Proceed to Phase 6 &nbsp;&nbsp; ☐ Remediate and re-validate


---
## Phase 6 — Scorecard Scaling and Governance Output

**Purpose:** Produce the final scored output and all governance documentation. This phase converts the validated model into business-usable outputs and packages the evidence required for model governance committee sign-off.


In [ ]:
# Selected interaction model output
# A fixed single points table is not valid for interaction terms because the
# deal-variable contribution depends on Equifax score. Use the Phase 4
# contribution table plus this coefficient table for governance output.

print(f'Selected model: {selected_model.model_name}')
print(selected_model.coefficient_table().to_string(index=False))


In [ ]:
# ── Score the development and OOT populations ────────────────────────────────
from modelling.scorecard_scaler import ScorecardScaler

# For the interaction model use predicted PD converted to score
scaler = ScorecardScaler(pdo=PDO, base_score=BASE_SCORE, base_odds=BASE_ODDS)

scores_dev = pd.Series([
    scaler.pd_to_score(p) for p in y_pred_dev
], index=dev_df.index, name='score')

scores_oot = pd.Series([
    scaler.pd_to_score(p) for p in y_pred_oot
], index=oot_df.index, name='score')

print(f'Development score: mean={scores_dev.mean():.0f}, std={scores_dev.std():.0f}')
print(f'OOT score:         mean={scores_oot.mean():.0f}, std={scores_oot.std():.0f}')


In [ ]:
# ── Score distribution: development vs OOT ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=False)

for ax, scores, y_true, label in [
    (axes[0], scores_dev, dev_df[TARGET], 'Development'),
    (axes[1], scores_oot, oot_df[TARGET], 'OOT'),
]:
    ax.hist(scores[y_true == 0], bins=40, alpha=0.6, color='#27ae60',
            label='Goods', density=True)
    ax.hist(scores[y_true == 1], bins=40, alpha=0.6, color='#c0392b',
            label='Bads',  density=True)
    ax.set_title(f'Score Distribution — {label}')
    ax.set_xlabel('Scorecard Score')
    ax.set_ylabel('Density')
    ax.legend()

plt.suptitle('Scorecard Score Separation', fontsize=10)
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/phase6_score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Score → PD reference table ───────────────────────────────────────────────
score_ref = pd.DataFrame({
    'score': range(480, 721, 20),
})
score_ref['approx_pd'] = score_ref['score'].apply(
    lambda s: f"{scaler.score_to_pd(s):.2%}"
)
print(score_ref.to_string(index=False))


### Governance Documentation Checklist

Complete this checklist before submitting for model governance review.

**Phase 0 — Data:**
- ☐ Development and OOT sample sizes and bad rates documented
- ☐ Data leakage check completed and signed off
- ☐ Missing data treatment documented per variable

**Phase 1 — Variable Assessment:**
- ☐ IV table with all candidate variables (including excluded)
- ☐ WoE profiles for all selected variables
- ☐ Log-odds shape plots and transformation rationale per variable
- ☐ Phase 1 conclusions cell completed

**Phase 2 — Multiplicative Assumption:**
- ☐ Breslow-Day results table
- ☐ Odds ratio forest plot
- ☐ Stratum diagnostics for any FAIL (root cause documented)
- ☐ Decision on model structure (standard scorecard vs interaction model) documented

**Phase 3 — Configuration:**
- ☐ `DealVariableConfig` settings traceable to Phase 1 and Phase 2 findings
- ☐ Mode summary table saved
- ☐ All model diagnostics reviewed (coefficient signs, VIF, p-values)

**Phase 4 — Model Selection:**
- ☐ AIC/BIC comparison table
- ☐ LR test results for nested model pairs
- ☐ Discrimination table (Gini, KS — development and OOT)
- ☐ Governance note on interaction-only models (hierarchical principle)
- ☐ Selected model stated with rationale
- ☐ Point contribution table at P25/P50/P75 Equifax

**Phase 5 — Validation:**
- ☐ Full validation report
- ☐ Calibration plot (observed vs predicted)
- ☐ PSI and CSI results
- ☐ Any remediation actions documented

**Phase 6 — Output:**
- ☐ Scorecard points table
- ☐ Score distribution plots (dev and OOT)
- ☐ Score → PD reference table
- ☐ All plots saved to `outputs/plots/`


### Final Sign-off

> **[ANALYST NOTE]** Complete before submission.

**Model name and version:** *[name — vX.X]*

**Selected model:** *[model_name from Phase 4]*

**Term structure:** *[Full / other — with justification if not Full]*

**Input modes summary:**
- *[variable]* — continuous (log) / continuous (none) / WoE

**Validation summary:**
- Gini dev/OOT: *[X / X]* — drop: *[X]*
- PSI: *[X]* — *[Stable / Moderate / Significant]*
- HL p-value: *[X]* — *[Calibrated / Recalibrated]*

**Known limitations:**
- *[List any limitations, exceptions, or monitoring requirements]*

**Analyst sign-off:** *[Name — Date]*

**Reviewer sign-off:** *[Name — Date]*
